# ADNI Data Preprocessing Pipeline — Step-by-Step Demo

This notebook demonstrates the preprocessing pipeline used to prepare data from the **Alzheimer's Disease Neuroimaging Initiative (ADNI)** for downstream analysis.

We use the `ADNIPreprocess` class to walk through each stage of the pipeline on a **small representative subset** of the data, showing intermediate results and explaining _what_ each step does and _why_.

### Pipeline Overview

| Step | Method | Purpose |
|------|--------|---------|
| 1 | `load_data()` | Load the raw clinical and MRI CSV files |
| 2 | `encode_multihot_variables()` | One-hot encode pipe-delimited multi-response columns |
| 3 | `coerce_numeric_columns()` | Fix string-contaminated numeric columns |
| 4 | `compute_bmi()` | Derive standardized height, weight, and BMI |
| 5 | `filter_cn_subjects()` | Identify CN subjects with single vs. multiple diagnoses |
| 6 | `build_transition_cohort()` | Extract CN→MCI/AD transition cohort |
| 7 | `build_no_transition_cohort()` | Extract stable CN cohort |
| 8 | `match_cohorts()` | Propensity-match transition ↔ no-transition subjects |
| 9 | `select_features()` | Remove near-zero-variance features |
| 10 | `export_datasets()` | Save final datasets to CSV |

---
## Setup

Import the `ADNIPreprocess` class and configure logging so we can see informational messages from each pipeline step.

In [3]:
import logging
import pandas as pd
import numpy as np

from src.data_preprocessing import ADNIPreprocess

logging.basicConfig(level=logging.INFO, format='%(name)s — %(message)s')

# Instantiate the preprocessor pointing to the demo subset
preprocessor = ADNIPreprocess(
    data_path="data/All_Subjects_My_Table_03Jul2025.csv",
    mri_path="data/UCSFFSX7_20Jun2025.csv",
    output_dir="data/demo_output/",
    variance_threshold=0.01,
)
print("ADNIPreprocess instance created.")

ADNIPreprocess instance created.


---
## Step 1 — Load Data

The first step loads two CSV files:

- **Subjects table** (`data_df`): Contains clinical, demographic, cognitive, and biomarker variables for each subject at every visit.
- **MRI table** (`mri_df`): Contains FreeSurfer-derived structural MRI measurements (cortical thickness, volumes, etc.).

Each subject has **multiple rows** — one per study visit (screening, baseline, follow-up at 6, 12, 24, … months).

In [4]:
preprocessor.load_data()

print(f"Subjects table: {preprocessor.data_df.shape[0]} rows × {preprocessor.data_df.shape[1]} columns")
print(f"MRI table:      {preprocessor.mri_df.shape[0]} rows × {preprocessor.mri_df.shape[1]} columns")
print(f"\nUnique subjects: {preprocessor.data_df['subject_id'].nunique()}")
print(f"Unique visits:   {preprocessor.data_df['visit'].unique()}")

src.data_preprocessing — Loading data from /work/cniel/sw/singularity_containers/dcf-adni/project/data/All_Subjects_My_Table_03Jul2025.csv and /work/cniel/sw/singularity_containers/dcf-adni/project/data/UCSFFSX7_20Jun2025.csv
/work/cniel/sw/singularity_containers/dcf-adni/project/src/data_preprocessing.py:222: DtypeWarning: Columns (20,31,35,38,46,56,65,76,77,115,127) have mixed types. Specify dtype option on import or set low_memory=False.
  self.data_df = pd.read_csv(data_path)


Subjects table: 28815 rows × 128 columns
MRI table:      11394 rows × 347 columns

Unique subjects: 4983
Unique visits:   ['sc' 'f' 'bl' 'm12' 'm24' 'm36' 'm48' 'm60' 'm72' 'm84' 'm96' 'm108'
 'm120' 'm132' 'm66' 'm54' 'm42' 'm78' 'm138' 'm90' 'm156' 'm102' 'm30'
 'm180' 'm114' 'm126' 'm144' 'm216' 'm168' 'm150' 'm162' 'm210' 'm204'
 'm222' 'm228' 'm174' 'm06' 'uns1' 'm18' 'm186' 'm192' 'm198' 'AUT' 'nv'
 'm234' 'v06' 'v11' 'v21' 'v31' 'v51' 'init' 'y1' 'y2' 'v41' 'y4' 'y3'
 'scmri' 'm03' 'y5' 'v01' 'v02' 'v04' 'v05' 'tau' 'v03' '4_sc' '4_init'
 '4_m12' '4_bl' '4_m24' nan]


/work/cniel/sw/singularity_containers/dcf-adni/project/src/data_preprocessing.py:223: DtypeWarning: Columns (11,12,13,14,15,16,17,18,19,20) have mixed types. Specify dtype option on import or set low_memory=False.
  self.mri_df = pd.read_csv(mri_path)


In [5]:
# Preview the first few rows of the subjects table
preprocessor.data_df.head()

,subject_id,visit,HMHYPERT,GLUCOSE,VSHEIGHT,VSHTUNIT,VSWEIGHT,VSWTUNIT,MH16SMOK,MH16ASMOK,...,MHCUR,MHSTAB,subject_age,PTGENDER,research_group,GENOTYPE,subject_date,PTRTYR,PTENGSPK,PTNLANG
0,011_S_0002,sc,0.0,NaN,71.0,1.0,89.0,2.0,0.0,NaN,...,0.0,1.0,74.38,1.0,CN,3/3,2005-08-17,1989.0,NaN,NaN
1,011_S_0003,sc,1.0,NaN,69.0,1.0,74.0,2.0,1.0,NaN,...,1.0,1.0,81.30,1.0,AD,3/4,2005-08-18,1989.0,NaN,NaN
2,022_S_0004,sc,1.0,NaN,69.0,1.0,184.0,1.0,1.0,NaN,...,1.0,1.0,67.63,1.0,MCI,3/3,2005-08-18,NaN,NaN,NaN
3,011_S_0005,sc,0.0,NaN,71.0,1.0,88.0,2.0,1.0,NaN,...,1.0,1.0,73.73,1.0,CN,3/3,2005-08-23,1987.0,NaN,NaN
4,022_S_0007,sc,1.0,NaN,64.0,1.0,179.0,1.0,1.0,NaN,...,0.0,1.0,75.40,1.0,AD,3/4,2005-05-25,1995.0,NaN,NaN


---
## Step 2 — Multi-Hot Encoding

Several ADNI columns store **multiple responses in a single cell** using pipe-delimited values. For example, the `KEYMED` column (key medications) might contain `"1|4|6"` meaning the patient is on Aricept (1), Namenda (4), and anti-depressants (6).

This step **one-hot encodes** each of these columns, expanding them into separate binary columns:

| Original `KEYMED` | → `KEYMED_0` | `KEYMED_1` | `KEYMED_4` | `KEYMED_6` | ... |
|:------------------:|:---:|:---:|:---:|:---:|:---:|
| `"1\|4\|6"`           | 0   | 1   | 1   | 1   | ... |
| `"0"`               | 1   | 0   | 0   | 0   | ... |
| `NaN`              | NaN | NaN | NaN | NaN | ... |

**Encoded variables:** `DXMOTHET`, `KEYMED`, `PTNLANG`, `MHNUM`, `PTMARRY`, `PTHOME`, `PTNOTRT`, `PTPLANG`

In [6]:
n_cols_before = preprocessor.data_df.shape[1]

# Show a sample of the raw KEYMED column before encoding
print("Before encoding — sample KEYMED values:")
print(preprocessor.data_df['KEYMED'].dropna().head(10).tolist())

preprocessor.encode_multihot_variables()

n_cols_after = preprocessor.data_df.shape[1]
print(f"\nColumns before: {n_cols_before}  →  after: {n_cols_after}  (+{n_cols_after - n_cols_before} new one-hot columns)")

src.data_preprocessing — Encoding multi-hot variables: ['DXMOTHET', 'KEYMED', 'PTNLANG', 'MHNUM', 'PTMARRY', 'PTHOME', 'PTNOTRT', 'PTPLANG']
src.data_preprocessing — (DXMOTHET) Unique values: [1, 2, 4, 5, 6, 7, 9, 11, 12, 14]
src.data_preprocessing — (KEYMED) Unique values: [0, 1, 2, 3, 4, 5, 6, 7]


Before encoding — sample KEYMED values:
['1|4', '6|7', '0', '0', '0', '0', '1|4', '5', '1|7', '0']


src.data_preprocessing — (PTNLANG) Unique values: [1, 2, 3]
src.data_preprocessing — (MHNUM) Unique values: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
src.data_preprocessing — (PTMARRY) Unique values: [1, 2, 3, 4, 5, 6]
src.data_preprocessing — (PTHOME) Unique values: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
src.data_preprocessing — (PTNOTRT) Unique values: [0, 1, 2]
src.data_preprocessing — (PTPLANG) Unique values: [1, 2, 3]



Columns before: 128  →  after: 190  (+62 new one-hot columns)


In [7]:
# Show the newly created KEYMED one-hot columns
keymed_cols = [c for c in preprocessor.data_df.columns if c.startswith('KEYMED_')]
print(f"KEYMED expanded into {len(keymed_cols)} binary columns: {keymed_cols}")
preprocessor.data_df[['KEYMED'] + keymed_cols].dropna().head()

KEYMED expanded into 8 binary columns: ['KEYMED_0', 'KEYMED_1', 'KEYMED_2', 'KEYMED_3', 'KEYMED_4', 'KEYMED_5', 'KEYMED_6', 'KEYMED_7']


,KEYMED,KEYMED_0,KEYMED_1,KEYMED_2,KEYMED_3,KEYMED_4,KEYMED_5,KEYMED_6,KEYMED_7
1133,1|4,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
1134,6|7,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
1135,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1137,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1138,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


---
## Step 3 — Numeric Coercion

Some columns that should be numeric contain **string artifacts** (e.g., `">1700"` or `"<5"` in lab results). This step forces those columns to numeric, converting unparseable strings to `NaN`.

Additionally, the **CV** (Coefficient of Variation) column contains values like `"12.5%"` — the trailing `%` is stripped and the result converted to float.

**Affected columns:** `RCT20`, `RCT1408`, `RCT19`, `HMT40`, `BAT126`, `RCT392`, `RCT14`, `CV`

In [8]:
# Show sample CV values before coercion
print("Before coercion — sample CV values:")
print(preprocessor.data_df['CV'].dropna().head(5).tolist())

print("\nBefore coercion — sample RCT20 values:")
print(preprocessor.data_df['RCT20'].dropna().head(5).tolist())

preprocessor.coerce_numeric_columns()

print("\nAfter coercion — sample CV values (float):")
print(preprocessor.data_df['CV'].dropna().head(5).tolist())

print("\nAfter coercion — sample RCT20 values (float):")
print(preprocessor.data_df['RCT20'].dropna().head(5).tolist())

src.data_preprocessing — Coercing numeric columns


Before coercion — sample CV values:
['2%', '9%', '9%', '0%', '2%']

Before coercion — sample RCT20 values:
['119', '181', '160', '163', '243']

After coercion — sample CV values (float):
[2.0, 9.0, 9.0, 0.0, 2.0]

After coercion — sample RCT20 values (float):
[119.0, 181.0, 160.0, 163.0, 243.0]


---
## Step 4 — BMI Computation

ADNI records height and weight in **mixed units** (imperial or metric), indicated by unit columns (`VSHTUNIT`, `VSWTUNIT`).

This step:
1. Converts all heights to **centimeters** (inches × 2.54 if imperial)
2. Converts all weights to **kilograms** (lbs × 0.4536 if imperial)
3. Computes **BMI** = weight(kg) / (height(m))²

Three new columns are added: `HEIGHT`, `WEIGHT`, `BMI`.

In [9]:
preprocessor.compute_bmi()

bmi_cols = ['VSHEIGHT', 'VSHTUNIT', 'HEIGHT', 'VSWEIGHT', 'VSWTUNIT', 'WEIGHT', 'BMI']
preprocessor.data_df[bmi_cols].dropna().head(10)

src.data_preprocessing — Computing BMI


,VSHEIGHT,VSHTUNIT,HEIGHT,VSWEIGHT,VSWTUNIT,WEIGHT,BMI
0,71.0,1.0,180.34,89.0,2.0,89.000000,27.365657
1,69.0,1.0,175.26,74.0,2.0,74.000000,24.091626
2,69.0,1.0,175.26,184.0,1.0,83.460928,27.171749
3,71.0,1.0,180.34,88.0,2.0,88.000000,27.058178
4,64.0,1.0,162.56,179.0,1.0,81.192968,30.724939
5,54.0,1.0,137.16,122.0,1.0,55.338224,29.415096
6,65.0,1.0,165.10,60.0,2.0,60.000000,22.011878
7,62.0,1.0,157.48,138.6,1.0,62.867851,25.349991
8,66.0,1.0,167.64,207.0,1.0,93.893544,33.410304
9,57.0,1.0,144.78,124.0,1.0,56.245408,26.833043


In [10]:
# Summary statistics for BMI
print("BMI distribution:")
preprocessor.data_df['BMI'].describe()

BMI distribution:


count    4981.000000
mean       27.544413
std         8.509549
min         2.793724
25%        23.821501
50%        26.516636
75%        29.885836
max       328.903405
Name: BMI, dtype: float64

---
## Step 5 — Filter Cognitively Normal (CN) Subjects

We focus on subjects whose **initial research group** was **Cognitively Normal (CN)**.

These CN subjects are split into two groups based on their longitudinal diagnostic trajectory:

| Group | Description | Clinical meaning |
|-------|-------------|------------------|
| **Multiple diagnoses** | Subject received ≥2 distinct diagnoses across visits | Potentially transitioned from CN → MCI or CN → Dementia |
| **Single diagnosis** | Subject always received the same diagnosis | Remained cognitively stable throughout the study |

This classification is the first step toward building the **transition** and **no-transition** cohorts.

In [11]:
preprocessor.filter_cn_subjects()

print(f"CN subjects with MULTIPLE diagnoses: {len(preprocessor._subjects_multiple_dx)}")
print(f"CN subjects with SINGLE diagnosis:   {len(preprocessor._subjects_one_dx)}")

src.data_preprocessing — Filtering CN subjects
src.data_preprocessing — CN subjects — multiple diagnoses: 163, single diagnosis: 1211


CN subjects with MULTIPLE diagnoses: 163
CN subjects with SINGLE diagnosis:   1211


In [12]:
# Example: show the diagnostic trajectory of a multi-diagnosis CN subject
example_subj = preprocessor._subjects_multiple_dx[0]
subj_df = preprocessor.data_df[preprocessor.data_df['subject_id'] == example_subj]
print(f"Subject: {example_subj}")
print(f"Research group: {subj_df['research_group'].iloc[0]}")
subj_df[['visit', 'DIAGNOSIS']].dropna()

Subject: 011_S_0002
Research group: CN


,visit,DIAGNOSIS
6769,m144,2.0
8076,bl,1.0
8124,m06,1.0
10254,m36,1.0
11060,m60,1.0
11418,m72,1.0
12194,m84,2.0
13095,m96,1.0
14221,m120,1.0
14580,m132,2.0


---
## Step 6 — Build the Transition Cohort

From the CN subjects with multiple diagnoses, we apply stricter **inclusion criteria** to build the _transition cohort_:

1. The subject must have **remained CN for at least the first 12 months** of the study — this ensures the baseline measurements were taken while the subject was still cognitively normal.
2. The subject's **last recorded diagnosis must be MCI (2) or Dementia (3)** — confirming an actual cognitive decline.

For each qualifying subject, we extract the **baseline measurements** (screening + baseline visits, back-filled/forward-filled to handle missing data) along with:
- `study_duration`: total months of follow-up
- `last_diagnosis`: the final diagnosis (MCI=2 or Dementia=3)
- `transition`: set to **1** (indicating this subject transitioned)

In [13]:
preprocessor.build_transition_cohort()

print(f"Transition cohort size: {preprocessor.subjects_transition_df.shape[0]} subjects")
print(f"\nLast diagnosis distribution (2=MCI, 3=Dementia):")
print(preprocessor.subjects_transition_df['last_diagnosis'].value_counts())
print(f"\nStudy duration (months) — summary:")
print(preprocessor.subjects_transition_df['study_duration'].describe())

src.data_preprocessing — Building transition cohort
src.data_preprocessing — Transition cohort: 127 subjects


Transition cohort size: 127 subjects

Last diagnosis distribution (2=MCI, 3=Dementia):
last_diagnosis
2.0    91
3.0    36
Name: count, dtype: int64

Study duration (months) — summary:
count    127.00000
mean     104.88189
std       48.98309
min       24.00000
25%       72.00000
50%      102.00000
75%      144.00000
max      228.00000
Name: study_duration, dtype: float64


In [14]:
# Preview transition cohort
preprocessor.subjects_transition_df[[
    'subject_id', 'subject_age', 'PTGENDER', 'GENOTYPE',
    'DIAGNOSIS', 'last_diagnosis', 'study_duration', 'transition'
]]

,subject_id,subject_age,PTGENDER,GENOTYPE,DIAGNOSIS,last_diagnosis,study_duration,transition
8076,011_S_0002,74.38,1,3/3,1,2.0,144,1
3454,011_S_0008,84.50,2,2/3,1,2.0,120,1
8078,100_S_0015,80.87,1,3/4,1,2.0,78,1
3456,011_S_0022,63.24,1,3/4,1,2.0,24,1
8081,100_S_0035,76.95,1,3/3,1,2.0,192,1
...,...,...,...,...,...,...,...,...
6296,016_S_6834,82.36,2,3/3,1,2.0,30,1
16136,014_S_6920,65.37,2,3/3,1,2.0,48,1
16170,014_S_6988,83.32,2,4/4,1,2.0,42,1
16187,941_S_7051,63.76,1,2/3,1,2.0,36,1


---
## Step 7 — Build the No-Transition Cohort

From CN subjects with a **single diagnosis** throughout all visits, we extract the same baseline information.

These subjects serve as the **control pool** — they remained cognitively normal for the entire study period. We extract more subjects than needed here because the next step (matching) will select the best matches and leave the rest as a separate test set.

Each subject gets:
- `study_duration`: total months of follow-up
- `last_diagnosis`: confirming CN (1)
- `transition`: set to **0**

In [15]:
preprocessor.build_no_transition_cohort()

print(f"No-transition cohort size: {preprocessor.subjects_no_transition_df.shape[0]} subjects")
print(f"\nAll have diagnosis = 1 (CN)? {(preprocessor.subjects_no_transition_df['last_diagnosis'] == 1).all()}")

src.data_preprocessing — Building no-transition cohort
src.data_preprocessing — No-transition cohort: 547 subjects


No-transition cohort size: 547 subjects

All have diagnosis = 1 (CN)? True


In [16]:
# Preview no-transition cohort
preprocessor.subjects_no_transition_df[[
    'subject_id', 'subject_age', 'PTGENDER', 'GENOTYPE',
    'DIAGNOSIS', 'last_diagnosis', 'study_duration', 'transition'
]]

,subject_id,subject_age,PTGENDER,GENOTYPE,DIAGNOSIS,last_diagnosis,study_duration,transition
3452,011_S_0005,73.73,1,3/3,1,1.0,36,0
3755,022_S_0014,78.54,2,3/3,1,1.0,36,0
3455,011_S_0016,65.45,1,3/4,1,1.0,36,0
3881,067_S_0019,73.14,2,2/3,1,1.0,36,0
8080,011_S_0021,72.65,2,2/3,1,1.0,228,0
...,...,...,...,...,...,...,...,...
15397,021_S_7092,68.56,1,3/3,1,1.0,24,0
15417,033_S_7100,72.93,1,4/4,1,1.0,24,0
15442,033_S_7114,65.69,2,3/4,1,1.0,24,0
16198,082_S_7122,67.30,2,nan,1,1.0,24,0


---
## Step 8 — Cohort Matching

To create a **balanced case-control study**, we pair each transition subject with the **best-matching** control (no-transition) subject.

### Matching criteria

| Criterion | Type | Strategy |
|-----------|------|----------|
| **Age** (`subject_age`) | Numeric | Minimize Euclidean distance |
| **Sex** (`PTGENDER`) | Categorical | Exact match required |
| **ApoE Genotype** (`GENOTYPE`) | Categorical | Exact match required |

### Algorithm
1. For each transition subject, compute the Euclidean distance (on `subject_age`) to all remaining controls.
2. Filter to only controls with **matching sex and genotype**.
3. Select the **closest** match.
4. Remove the matched control from the pool (matching **without replacement**).

The result is:
- **`joint_dataset_df`**: the paired dataset (transition + matched controls), with a `group` column linking each pair.
- **`remaining_test_df`**: unmatched controls, usable as a held-out test set.

In [17]:
preprocessor.match_cohorts()

print(f"Joint dataset:     {preprocessor.joint_dataset_df.shape[0]} subjects ({preprocessor.joint_dataset_df.shape[0]//2} matched pairs)")
print(f"Remaining test:    {preprocessor.remaining_test_df.shape[0]} unmatched controls")
print(f"\nTransition distribution in joint dataset:")
print(preprocessor.joint_dataset_df['transition'].value_counts())

src.data_preprocessing — Matching transition ↔ no-transition cohorts
src.data_preprocessing — Matched 127 pairs; 420 remaining unmatched controls


Joint dataset:     254 subjects (127 matched pairs)
Remaining test:    420 unmatched controls

Transition distribution in joint dataset:
transition
1    127
0    127
Name: count, dtype: int64


In [18]:
# Show the matched pairs side by side
match_cols = ['subject_id', 'subject_age', 'PTGENDER', 'GENOTYPE', 'transition', 'group']
pairs_df = preprocessor.joint_dataset_df[match_cols].sort_values(by=['group', 'transition'])
pairs_df

,subject_id,subject_age,PTGENDER,GENOTYPE,transition,group
4733,072_S_4391,74.47,1,3/3,0,1
8076,011_S_0002,74.38,1,3/3,1,1
8226,094_S_0526,85.09,2,2/3,0,2
3454,011_S_0008,84.50,2,2/3,1,2
3662,141_S_0726,81.01,1,3/4,0,3
...,...,...,...,...,...,...
16170,014_S_6988,83.32,2,4/4,1,125
6356,021_S_6896,64.79,1,2/3,0,126
16187,941_S_7051,63.76,1,2/3,1,126
5889,003_S_6067,63.12,2,3/3,0,127


In [19]:
# Verify matching quality: age difference within each pair
transition_ages = preprocessor.subjects_transition_df.set_index('group')['subject_age'].rename('age_transition')
control_ages = preprocessor.subjects_pairs_df.set_index('group')['subject_age'].rename('age_control')
pair_comparison = pd.concat([transition_ages, control_ages], axis=1)
pair_comparison['age_diff'] = (pair_comparison['age_transition'] - pair_comparison['age_control']).abs()
print("Age difference per matched pair:")
print(pair_comparison)
print(f"\nMean age difference: {pair_comparison['age_diff'].mean():.2f} years")

Age difference per matched pair:
       age_transition  age_control  age_diff
group                                       
1               74.38        74.47      0.09
2               84.50        85.09      0.59
3               80.87        81.01      0.14
4               63.24        63.21      0.03
5               76.95        76.97      0.02
...               ...          ...       ...
123             82.36        81.71      0.65
124             65.37        65.37      0.00
125             83.32        67.29     16.03
126             63.76        64.79      1.03
127             62.95        63.12      0.17

[127 rows x 3 columns]

Mean age difference: 0.50 years


---
## Step 9 — Feature Selection

With all subjects assembled, we apply **variance-based feature selection** using scikit-learn's `VarianceThreshold`.

Features with variance below the threshold are removed — these are columns that are nearly constant across all subjects and carry little discriminative information.

**Categorical/metadata columns** (e.g., `subject_id`, `GENOTYPE`, `visit`) are excluded from this filter and always retained.

In [20]:
n_before = preprocessor.joint_dataset_df.shape[1]

preprocessor.select_features()

n_after = preprocessor.joint_dataset_df.shape[1]
print(f"Features before selection: {n_before}")
print(f"Features after selection:  {n_after}")
print(f"Removed: {n_before - n_after} near-zero-variance features")
print(f"\nVariance threshold used: {preprocessor.variance_threshold}")

src.data_preprocessing — Selecting features with variance > 0.01
/usr/local/lib/python3.12/site-packages/sklearn/feature_selection/_variance_threshold.py:114: RuntimeWarning: Degrees of freedom <= 0 for slice.
  self.variances_ = np.nanvar(X, axis=0)
src.data_preprocessing — Keeping 124 features


Features before selection: 197
Features after selection:  124
Removed: 73 near-zero-variance features

Variance threshold used: 0.01


In [21]:
# Show kept feature names
print(f"Retained features ({len(preprocessor.keep_features)}):")
print(sorted(preprocessor.keep_features))

Retained features (124):
[np.str_('ABETA42'), np.str_('BAT126'), np.str_('BCCHEST'), np.str_('BCDPMOOD'), np.str_('BCFALL'), np.str_('BCHDACHE'), np.str_('BCINSOMN'), np.str_('BCMUSCLE'), np.str_('BCVISION'), np.str_('BMI'), np.str_('BSXCHRON'), np.str_('BSXSEVER'), np.str_('BSXSYMNO'), np.str_('CDRSB'), np.str_('DXCONFID'), np.str_('DXDEP'), np.str_('FAQFINAN'), np.str_('FAQGAME'), np.str_('FAQSHOP'), np.str_('FAQTOTAL'), np.str_('FAQTRAVL'), np.str_('GDTOTAL'), 'GENOTYPE', np.str_('GLUCOSE'), np.str_('HEIGHT'), np.str_('HMHYPERT'), np.str_('HMSCORE'), np.str_('HMSOMATC'), np.str_('HMSTROKE'), np.str_('HMT40'), np.str_('KEYMED_0'), np.str_('KEYMED_6'), np.str_('KEYMED_7'), np.str_('LDELTOTAL'), np.str_('LIMMTOTAL'), np.str_('MH12RENA'), np.str_('MH14AALCH'), np.str_('MH14ALCH'), np.str_('MH16ASMOK'), np.str_('MH16CSMOK'), np.str_('MH16SMOK'), np.str_('MH4CARD'), np.str_('MHCUR'), np.str_('MHNUM'), np.str_('MHNUM_1'), np.str_('MHNUM_10'), np.str_('MHNUM_13'), np.str_('MHNUM_18'), np.st

---
## Step 10 — Export Datasets

The final step saves three output CSV files:

| File | Contents |
|------|----------|
| `joint_dataset.csv` | Matched transition + control pairs (the main analysis dataset) |
| `remaining_test.csv` | Unmatched control subjects (held-out test set) |
| `mri_joint_dataset.csv` | MRI measurements for subjects in the joint dataset |

In [ ]:
preprocessor.export_datasets()

print("\nDone! Output files saved to:", preprocessor.output_dir)

---
## Summary

The `ADNIPreprocess` pipeline transforms raw ADNI data into a clean, analysis-ready dataset through the following stages:

```
Raw CSVs
 │
 ├─ 1. Load data  ──────────────────  2 CSV files → DataFrames
 ├─ 2. Multi-hot encoding  ─────────  Pipe-delimited → binary columns
 ├─ 3. Numeric coercion  ───────────  Fix string artifacts in lab values
 ├─ 4. BMI computation  ────────────  Standardize units → derive BMI
 ├─ 5. Filter CN subjects  ─────────  Split by diagnostic trajectory
 ├─ 6. Transition cohort  ──────────  CN→MCI/AD with 12-month rule
 ├─ 7. No-transition cohort  ───────  Stable CN controls
 ├─ 8. Cohort matching  ────────────  1:1 age/sex/genotype matching
 ├─ 9. Feature selection  ──────────  Remove near-constant features
 └─ 10. Export  ────────────────────  3 output CSVs
```

All steps are encapsulated in the `ADNIPreprocess` class and can be run end-to-end via `preprocessor.run()`, or step-by-step as shown in this notebook.